# Detecção de placas de sinalização — RoboCar

Notebook de treinamento e avaliação. Roda de cima a baixo.

- **Bloco 0** — setup, constantes e vídeo de referência
- **Bloco 1** — dataset próprio: download, contagem, treino, avaliação
- **Bloco 2** — dataset público: download, filtro, treino, avaliação cruzada
- **Bloco 3** — datasets descartados: apenas contagem
- **Bloco 4** — empacota tudo para download

## Bloco 0 — Setup

In [ ]:
!pip install -q ultralytics roboflow

import torch, ultralytics
print("ultralytics:", ultralytics.__version__)
print("torch      :", torch.__version__)
print("GPU        :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 5.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics: 8.4.121
torch      : 2.11.0+cu128
GPU        : Tesla T4


In [ ]:
from pathlib import Path

CLASSES_FINAIS = [
    "R-25a_esquerda",
    "R-25b_direita",
    "R-26_frente",
    "R-3_sentido-proibido",
    "R-4a_proibido-esquerda",
    "R-4b_proibido-direita",
]

RF_KEY = "aVtYS205nG9XQsuNywb6"

Path("/content/docs").mkdir(exist_ok=True)

In [ ]:
!wget -q -O /content/video_robocar.mp4 "https://drive.google.com/file/d/1lEnlrQsOOebodOIKd7LiijreMVIVnNp_/view?usp=sharing"


In [ ]:
import requests

file_id = '1lEnlrQsOOebodOIKd7LiijreMVIVnNp_'
download_url = f'https://drive.google.com/uc?export=download&id={file_id}'

r = requests.get(download_url, stream=True)
with open('/content/video_robocar.mp4', 'wb') as f:
    for chunk in r.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)

!ls -lh {"/content/video_robocar.mp4"}

-rw-r--r-- 1 root root 16M Aug 17 20:44 /content/video_robocar.mp4


## Bloco 1 — Dataset próprio

Imagens capturadas pela ESP32-CAM em cinco sessões. Split por sessão:
S1–S3 treino, S4 validação, S5 teste.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=RF_KEY)
ds = rf.workspace("vitor-chagas").project("robocar-dataset-c").version(2).download("yolov8")
RAIZ_A = Path(ds.location)
root = RAIZ_A
print(root)

loading Roboflow workspace...
loading Roboflow project...
/content/robocar-dataset-C-2


In [ ]:
from collections import Counter
import yaml

names = yaml.safe_load(open(root/"data.yaml"))["names"]
for split in ["train", "valid", "test"]:
    if not (root/split/"images").exists():
        continue
    c = Counter()
    n_imgs = n_fundo = 0
    # Percorre as IMAGENS, nao os labels: imagem de fundo pode nao ter .txt
    for img in (root/split/"images").iterdir():
        n_imgs += 1
        txt = root/split/"labels"/(img.stem + ".txt")
        linhas = txt.read_text().splitlines() if txt.exists() else []
        linhas = [l for l in linhas if l.strip()]
        if not linhas:
            n_fundo += 1
        for linha in linhas:
            c[int(linha.split()[0])] += 1
    print(f"\n--- {split} ---")
    print(f"imagens          : {n_imgs}")
    print(f"imagens de fundo : {n_fundo}")
    print(f"anotacoes        : {sum(c.values())}")
    for i, nome in enumerate(names):
        print(f"  {c[i]:5d}  {nome}")


--- train ---
imagens          : 645
imagens de fundo : 85
anotacoes        : 695
    130  R-25a_esquerda
    125  R-25b_direita
    119  R-26_frente
    107  R-3_sentido-proibido
    106  R-4a_proibido-esquerda
    108  R-4b_proibido-direita

--- valid ---
imagens          : 318
imagens de fundo : 42
anotacoes        : 326
     51  R-25a_esquerda
     64  R-25b_direita
     46  R-26_frente
     44  R-3_sentido-proibido
     67  R-4a_proibido-esquerda
     54  R-4b_proibido-direita

--- test ---
imagens          : 269
imagens de fundo : 36
anotacoes        : 321
     62  R-25a_esquerda
     71  R-25b_direita
     45  R-26_frente
     55  R-3_sentido-proibido
     44  R-4a_proibido-esquerda
     44  R-4b_proibido-direita


In [ ]:
# Confere que a ordem das classes bate com a ordem canonica.

print("dataset :", list(names))
print("canonica:", CLASSES_FINAIS)
print("BATE" if list(names) == CLASSES_FINAIS else "NAO BATE - corrigir antes de seguir")

dataset : ['R-25a_esquerda', 'R-25b_direita', 'R-26_frente', 'R-3_sentido-proibido', 'R-4a_proibido-esquerda', 'R-4b_proibido-direita']
canonica: ['R-25a_esquerda', 'R-25b_direita', 'R-26_frente', 'R-3_sentido-proibido', 'R-4a_proibido-esquerda', 'R-4b_proibido-direita']
BATE


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")   # pesos COCO, transfer learning

model.train(
    data=str(RAIZ_A/"data.yaml"),
    epochs=150,
    imgsz=640,
    batch=16,
    seed=0,
    fliplr=0.0,        # espelhar trocaria esquerda por direita
    degrees=8.0,       # placa levemente torta e realista; muito nao
    hsv_v=0.4,         # brilho: a variacao real do dominio
    hsv_s=0.5,         # saturacao: temperatura de cor entre comodos
    patience=30,
    project="/content/runs",
    name="A_v11n",
)

Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.13.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/robocar-dataset-C-2/data.yaml, degrees=8.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=A_proprio-2, nbs=64, n

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7acf3ab1e1e0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [ ]:
from ultralytics import YOLO
# Avaliacao no conjunto de teste (sessao 5, nunca vista no treino).
melhor_a = YOLO("/content/runs/A_v11n/weights/best.pt")
val_a = melhor_a.val(data=str(RAIZ_A/"data.yaml"), split="test",
                     project="/content/runs", name="A_val_teste")

print("\n=== modelo A no teste proprio ===")
print(f"mAP@0.5      : {val_a.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {val_a.box.map:.4f}")
print(f"precisao     : {val_a.box.mp:.4f}")
print(f"revocacao    : {val_a.box.mr:.4f}")
print("\npor classe:")
for i, nome in enumerate(CLASSES_FINAIS):
    p, r, _, _ = val_a.box.class_result(i)
    f1 = 2*p*r/(p+r) if (p+r) else 0.0
    print(f"  {nome:26s} P={p:.3f}  R={r:.3f}  F1={f1:.3f}  mAP50={val_a.box.ap50[i]:.3f}")

Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.13.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 613.5±256.3 MB/s, size: 14.7 KB)
val: Scanning /content/robocar-dataset-C-2/test/labels.cache... 269 images, 36 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 269/269 112.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 5.2it/s 3.3s
                   all        269        321      0.989      0.995      0.983      0.974
        R-25a_esquerda         62         62      0.996      0.968      0.965      0.953
         R-25b_direita         71         71      0.998          1      0.995      0.989
           R-26_frente         45         45      0.994          1      0.995      0.987
  R-3_sentido-proibido         55         55      0.996          1      0.995      0.988
R-4a_proibido-esquerda         44

In [ ]:
# stream=True processa frame a frame, sem acumular os resultados na memoria.
# Sem isso, o predict guarda todos os frames e estoura a RAM do Colab.
resultados = melhor_a.predict(
    source='/content/video_robocar.mp4',
    conf=0.5,
    save=True,
    stream=True,
    project="/content/runs",
    name="A_video",
)

# com stream=True o predict e preguicoso: precisa consumir o gerador
for _ in resultados:
    pass

print("video gerado")


video 1/1 (frame 1/2245) /content/video_robocar.mp4: 384x640 1 R-4a_proibido-esquerda, 8.9ms
video 1/1 (frame 2/2245) /content/video_robocar.mp4: 384x640 1 R-4a_proibido-esquerda, 9.0ms
video 1/1 (frame 3/2245) /content/video_robocar.mp4: 384x640 1 R-4a_proibido-esquerda, 9.3ms
video 1/1 (frame 4/2245) /content/video_robocar.mp4: 384x640 1 R-4a_proibido-esquerda, 11.5ms
video 1/1 (frame 5/2245) /content/video_robocar.mp4: 384x640 1 R-4a_proibido-esquerda, 9.1ms
video 1/1 (frame 6/2245) /content/video_robocar.mp4: 384x640 1 R-4a_proibido-esquerda, 9.6ms
video 1/1 (frame 7/2245) /content/video_robocar.mp4: 384x640 1 R-4a_proibido-esquerda, 9.6ms
video 1/1 (frame 8/2245) /content/video_robocar.mp4: 384x640 1 R-4a_proibido-esquerda, 8.7ms
video 1/1 (frame 9/2245) /content/video_robocar.mp4: 384x640 1 R-4a_proibido-esquerda, 9.4ms
video 1/1 (frame 10/2245) /content/video_robocar.mp4: 384x640 1 R-4a_proibido-esquerda, 8.1ms
video 1/1 (frame 11/2245) /content/video_robocar.mp4: 384x640 1 R-4

## Bloco 2 — Dataset público (Ramo B)

`normal-brazilian-traffic-signs`, filtrado para as 6 classes de interesse.
O modelo resultante é avaliado **no conjunto de teste do dataset próprio**:
é isso que mede se o dado público generaliza para o domínio de operação.

In [ ]:
ds = rf.workspace("vitor-chagas").project("normal-brazilian-traffic-signs-88eft").version(1).download("yolov8")
RAIZ_PUB = Path(ds.location)
root = RAIZ_PUB
print(root)

loading Roboflow workspace...
loading Roboflow project...
/content/normal-brazilian-traffic-signs-1


In [ ]:
from collections import Counter
import yaml

names = yaml.safe_load(open(root/"data.yaml"))["names"]
for split in ["train", "valid", "test"]:
    if not (root/split/"images").exists():
        continue
    c = Counter()
    n_imgs = n_fundo = 0
    # Percorre as IMAGENS, nao os labels: imagem de fundo pode nao ter .txt
    for img in (root/split/"images").iterdir():
        n_imgs += 1
        txt = root/split/"labels"/(img.stem + ".txt")
        linhas = txt.read_text().splitlines() if txt.exists() else []
        linhas = [l for l in linhas if l.strip()]
        if not linhas:
            n_fundo += 1
        for linha in linhas:
            c[int(linha.split()[0])] += 1
    print(f"\n--- {split} ---")
    print(f"imagens          : {n_imgs}")
    print(f"imagens de fundo : {n_fundo}")
    print(f"anotacoes        : {sum(c.values())}")
    for i, nome in enumerate(names):
        print(f"  {c[i]:5d}  {nome}")


--- train ---
imagens          : 2816
imagens de fundo : 2662
anotacoes        : 156
     21  R-25a
     27  R-25b
     28  R-26
     28  R-3
     27  R-4a
     25  R-4b

--- valid ---
imagens          : 804
imagens de fundo : 759
anotacoes        : 45
      9  R-25a
      7  R-25b
      7  R-26
      6  R-3
      8  R-4a
      8  R-4b

--- test ---
imagens          : 402
imagens de fundo : 374
anotacoes        : 30
      8  R-25a
      5  R-25b
      3  R-26
      5  R-3
      4  R-4a
      5  R-4b


In [ ]:
# Mapa id_antigo -> id_novo, procurando o codigo da placa dentro do nome
# original nas duas grafias que aparecem: (R-26) e -R-26-
codigos = [c.split("_")[0] for c in CLASSES_FINAIS]   # R-25a, R-25b, ...

mapa = {}
for i, nome in enumerate(names):
    for novo_id, cod in enumerate(codigos):
        if f"{cod}" in nome:
            mapa[i] = novo_id

for antigo, novo in sorted(mapa.items()):
    print(f"{antigo:3d} {names[antigo]:45s} -> {novo} ({CLASSES_FINAIS[novo]})")

  0 R-25a                                         -> 0 (R-25a_esquerda)
  1 R-25b                                         -> 1 (R-25b_direita)
  2 R-26                                          -> 2 (R-26_frente)
  3 R-3                                           -> 3 (R-3_sentido-proibido)
  4 R-4a                                          -> 4 (R-4a_proibido-esquerda)
  5 R-4b                                          -> 5 (R-4b_proibido-direita)


In [ ]:
# Reescreve labels e copia imagens, so com as 6 classes e ja na ordem canonica.
import shutil

RAIZ_B = Path("/content/dataset6")
if RAIZ_B.exists():
    shutil.rmtree(RAIZ_B)

for split in ["train", "valid", "test"]:
    if not (RAIZ_PUB/split/"labels").exists():
        continue
    (RAIZ_B/split/"images").mkdir(parents=True, exist_ok=True)
    (RAIZ_B/split/"labels").mkdir(parents=True, exist_ok=True)
    mantidas = 0
    for txt in (RAIZ_PUB/split/"labels").glob("*.txt"):
        novas = []
        for linha in txt.read_text().splitlines():
            if not linha.strip():
                continue
            campos = linha.split()
            antigo = int(campos[0])
            if antigo in mapa:                 # so as 6 de interesse
                novas.append(" ".join([str(mapa[antigo])] + campos[1:]))
        if not novas:                          # sem objeto de interesse -> descarta
            continue
        img = next((RAIZ_PUB/split/"images").glob(txt.stem + ".*"), None)
        if img is None:
            continue
        (RAIZ_B/split/"labels"/txt.name).write_text("\n".join(novas))
        shutil.copy(img, RAIZ_B/split/"images"/img.name)
        mantidas += 1
    print(f"{split}: {mantidas} imagens mantidas")

(RAIZ_B/"data.yaml").write_text(
    f"path: {RAIZ_B}\ntrain: train/images\nval: valid/images\ntest: test/images\n\n"
    f"nc: 6\nnames: {CLASSES_FINAIS}\n"
)
root = RAIZ_B

train: 154 imagens mantidas
valid: 45 imagens mantidas
test: 28 imagens mantidas


In [ ]:
from collections import Counter
import yaml

names = yaml.safe_load(open(root/"data.yaml"))["names"]
for split in ["train", "valid", "test"]:
    if not (root/split/"images").exists():
        continue
    c = Counter()
    n_imgs = n_fundo = 0
    # Percorre as IMAGENS, nao os labels: imagem de fundo pode nao ter .txt
    for img in (root/split/"images").iterdir():
        n_imgs += 1
        txt = root/split/"labels"/(img.stem + ".txt")
        linhas = txt.read_text().splitlines() if txt.exists() else []
        linhas = [l for l in linhas if l.strip()]
        if not linhas:
            n_fundo += 1
        for linha in linhas:
            c[int(linha.split()[0])] += 1
    print(f"\n--- {split} ---")
    print(f"imagens          : {n_imgs}")
    print(f"imagens de fundo : {n_fundo}")
    print(f"anotacoes        : {sum(c.values())}")
    for i, nome in enumerate(names):
        print(f"  {c[i]:5d}  {nome}")


--- train ---
imagens          : 154
imagens de fundo : 0
anotacoes        : 156
     21  R-25a_esquerda
     27  R-25b_direita
     28  R-26_frente
     28  R-3_sentido-proibido
     27  R-4a_proibido-esquerda
     25  R-4b_proibido-direita

--- valid ---
imagens          : 45
imagens de fundo : 0
anotacoes        : 45
      9  R-25a_esquerda
      7  R-25b_direita
      7  R-26_frente
      6  R-3_sentido-proibido
      8  R-4a_proibido-esquerda
      8  R-4b_proibido-direita

--- test ---
imagens          : 28
imagens de fundo : 0
anotacoes        : 30
      8  R-25a_esquerda
      5  R-25b_direita
      3  R-26_frente
      5  R-3_sentido-proibido
      4  R-4a_proibido-esquerda
      5  R-4b_proibido-direita


In [ ]:
# Mesmos hiperparametros do Bloco 1: a comparacao so isola o efeito do
# dataset se todo o resto for identico.

model = YOLO("yolo11n.pt")

model.train(
    data=str(RAIZ_B/"data.yaml"),
    epochs=150,
    imgsz=640,
    batch=16,
    seed=0,
    fliplr=0.0,
    degrees=8.0,
    hsv_v=0.4,
    hsv_s=0.5,
    patience=30,
    project="/content/runs",
    name="B_v11n",
)

Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.13.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset6/data.yaml, degrees=8.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=B_publico-2, nbs=64, nms=False, o

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7acf3ac98710>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [ ]:
# A comparacao que importa: modelo treinado em dado publico, avaliado no
# conjunto de teste do dominio real. Mesmas imagens, mesmas anotacoes e
# mesma metrica do modelo A -- so o dado de treino muda.
melhor_b = YOLO("/content/runs/B_v11n/weights/best.pt")
val_b = melhor_b.val(data=str(RAIZ_A/"data.yaml"), split="test",
                     project="/content/runs", name="B_val_teste_proprio")

print("\n=== modelo B no teste proprio ===")
print(f"mAP@0.5      : {val_b.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {val_b.box.map:.4f}")
print(f"precisao     : {val_b.box.mp:.4f}")
print(f"revocacao    : {val_b.box.mr:.4f}")
print("\npor classe:")
for i, nome in enumerate(CLASSES_FINAIS):
    p, r, _, _ = val_b.box.class_result(i)
    f1 = 2*p*r/(p+r) if (p+r) else 0.0
    print(f"  {nome:26s} P={p:.3f}  R={r:.3f}  F1={f1:.3f}  mAP50={val_b.box.ap50[i]:.3f}")

print("\n=== comparacao (tabela do relatorio) ===")
print(f"{'modelo':<24}{'mAP@0.5':>10}{'mAP@0.5:0.95':>15}")
print(f"{'A (dataset proprio)':<24}{val_a.box.map50:>10.4f}{val_a.box.map:>15.4f}")
print(f"{'B (dataset publico)':<24}{val_b.box.map50:>10.4f}{val_b.box.map:>15.4f}")

Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.13.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 725.5±240.4 MB/s, size: 14.7 KB)
val: Scanning /content/robocar-dataset-C-2/test/labels.cache... 269 images, 36 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 269/269 94.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 5.2it/s 3.3s
                   all        269        321      0.749      0.429      0.438      0.381
        R-25a_esquerda         62         62          1      0.339      0.387      0.324
         R-25b_direita         71         71      0.877      0.902      0.959      0.843
           R-26_frente         45         45       0.97      0.244      0.256      0.229
  R-3_sentido-proibido         55         55      0.674      0.273      0.303      0.267
R-4a_proibido-esquerda         44 

In [ ]:
# stream=True processa frame a frame, sem acumular os resultados na memoria.
# Sem isso, o predict guarda todos os frames e estoura a RAM do Colab.
resultados = melhor_b.predict(
    source='/content/video_robocar.mp4',
    conf=0.5,
    save=True,
    stream=True,
    project="/content/runs",
    name="B_video",
)

# com stream=True o predict e preguicoso: precisa consumir o gerador
for _ in resultados:
    pass

print("video gerado")


video 1/1 (frame 1/2245) /content/video_robocar.mp4: 384x640 (no detections), 7.9ms
video 1/1 (frame 2/2245) /content/video_robocar.mp4: 384x640 (no detections), 10.5ms
video 1/1 (frame 3/2245) /content/video_robocar.mp4: 384x640 (no detections), 10.1ms
video 1/1 (frame 4/2245) /content/video_robocar.mp4: 384x640 (no detections), 9.0ms
video 1/1 (frame 5/2245) /content/video_robocar.mp4: 384x640 (no detections), 11.3ms
video 1/1 (frame 6/2245) /content/video_robocar.mp4: 384x640 (no detections), 11.4ms
video 1/1 (frame 7/2245) /content/video_robocar.mp4: 384x640 (no detections), 10.3ms
video 1/1 (frame 8/2245) /content/video_robocar.mp4: 384x640 (no detections), 10.8ms
video 1/1 (frame 9/2245) /content/video_robocar.mp4: 384x640 (no detections), 11.3ms
video 1/1 (frame 10/2245) /content/video_robocar.mp4: 384x640 (no detections), 10.0ms
video 1/1 (frame 11/2245) /content/video_robocar.mp4: 384x640 (no detections), 10.6ms
video 1/1 (frame 12/2245) /content/video_robocar.mp4: 384x640 (n

## Bloco 3 — Datasets descartados

Sem treino: a contagem por classe é a evidência do descarte.

In [ ]:
# brazilian-traffic-signs: descartado por desbalanceamento severo
ds = rf.workspace("projetos-abip9").project("brazilian-traffic-signs-hnifq").version(6).download("yolov8")
root = Path(ds.location)
print(root)

loading Roboflow workspace...
loading Roboflow project...
/content/Brazilian-Traffic-Signs-6


In [ ]:
from collections import Counter
import yaml

names = yaml.safe_load(open(root/"data.yaml"))["names"]
for split in ["train", "valid", "test"]:
    if not (root/split/"images").exists():
        continue
    c = Counter()
    n_imgs = n_fundo = 0
    # Percorre as IMAGENS, nao os labels: imagem de fundo pode nao ter .txt
    for img in (root/split/"images").iterdir():
        n_imgs += 1
        txt = root/split/"labels"/(img.stem + ".txt")
        linhas = txt.read_text().splitlines() if txt.exists() else []
        linhas = [l for l in linhas if l.strip()]
        if not linhas:
            n_fundo += 1
        for linha in linhas:
            c[int(linha.split()[0])] += 1
    print(f"\n--- {split} ---")
    print(f"imagens          : {n_imgs}")
    print(f"imagens de fundo : {n_fundo}")
    print(f"anotacoes        : {sum(c.values())}")
    for i, nome in enumerate(names):
        print(f"  {c[i]:5d}  {nome}")


--- train ---
imagens          : 4189
imagens de fundo : 95
anotacoes        : 11330
     32  (A-12) Rotatória
     96  (A-18) Lombada
      9  (A-1a) Curva acentuada à esquerda
     21  (A-1b) Curva acentuada à direita
     45  (A-24) Obras
     27  (A-2a) Curva à esquerda
     14  (A-2b) Curva à direita
      2  (A-30a) Trânsito de ciclistas
      3  (A-30b) Passagem sinalizada de ciclistas
     40  (A-32a) Trânsito de pedestres
    149  (A-32b) Passagem sinalizada de pedestres
     18  (A-33a) Área escolar
     12  (A-33b) Passagem sinalizada de escolares
     21  (A-36) Animais selvagens
      2  (A-3a) Pista sinuosa à esquerda
     16  (A-5b) Curva em "S" à direita
    117  (MA) Marcador de alinhamento
     36  (MP-1) Passagem pela direita
     33  (MP-2) Passagem pela esquerda
     56  (MP-3) Passagem dupla
     70  (R-1) Pare
     10  (R-12) Proibido trânsito de bicicletas
    106  (R-15) Altura máxima
     20  (R-19) 100km/h
     12  (R-19) 10km/h
     20  (R-19) 110km/h
     

In [ ]:
# Mapa id_antigo -> id_novo, procurando o codigo da placa dentro do nome
# original nas duas grafias que aparecem: (R-26) e -R-26-
names_pub = yaml.safe_load(open(RAIZ_PUB/"data.yaml"))["names"]
codigos = [c.split("_")[0] for c in CLASSES_FINAIS]   # R-25a, R-25b, ...

mapa = {}
for i, nome in enumerate(names_pub):
    for novo_id, cod in enumerate(codigos):
        if f"({cod})" in nome or f"-{cod}-" in nome:
            mapa[i] = novo_id

for antigo, novo in sorted(mapa.items()):
    print(f"{antigo:3d} {names_pub[antigo]:45s} -> {novo} ({CLASSES_FINAIS[novo]})")

In [ ]:
# placas-transito: descartado por cobertura (so classes de proibicao)
ds = rf.workspace("ia-axk16").project("placas-transito").version(10).download("yolov8")
root = Path(ds.location)
print(root)

loading Roboflow workspace...
loading Roboflow project...
/content/placas-transito-10


In [ ]:
from collections import Counter
import yaml

names = yaml.safe_load(open(root/"data.yaml"))["names"]
for split in ["train", "valid", "test"]:
    if not (root/split/"images").exists():
        continue
    c = Counter()
    n_imgs = n_fundo = 0
    # Percorre as IMAGENS, nao os labels: imagem de fundo pode nao ter .txt
    for img in (root/split/"images").iterdir():
        n_imgs += 1
        txt = root/split/"labels"/(img.stem + ".txt")
        linhas = txt.read_text().splitlines() if txt.exists() else []
        linhas = [l for l in linhas if l.strip()]
        if not linhas:
            n_fundo += 1
        for linha in linhas:
            c[int(linha.split()[0])] += 1
    print(f"\n--- {split} ---")
    print(f"imagens          : {n_imgs}")
    print(f"imagens de fundo : {n_fundo}")
    print(f"anotacoes        : {sum(c.values())}")
    for i, nome in enumerate(names):
        print(f"  {c[i]:5d}  {nome}")


--- train ---
imagens          : 1801
imagens de fundo : 4
anotacoes        : 2155
     99  NoEstacionar
     72  NoSeguir
     41  NoUltrapassar
    119  NoVirarDireita
    152  NoVirarEsquerda
    138  NoVirarU
     75  Pare
     51  Rotatoria
    156  preferencia
    158  speed_limit_100
     47  speed_limit_110
    139  speed_limit_120
     44  speed_limit_20
    110  speed_limit_30
    176  speed_limit_40
    216  speed_limit_50
    125  speed_limit_60
     51  speed_limit_70
    112  speed_limit_80
     74  speed_limit_90

--- valid ---
imagens          : 464
imagens de fundo : 2
anotacoes        : 584
     27  NoEstacionar
     16  NoSeguir
     10  NoUltrapassar
     21  NoVirarDireita
     44  NoVirarEsquerda
     43  NoVirarU
     20  Pare
     16  Rotatoria
     37  preferencia
     54  speed_limit_100
      7  speed_limit_110
     40  speed_limit_120
     10  speed_limit_20
     30  speed_limit_30
     46  speed_limit_40
     56  speed_limit_50
     41  speed_limit_60
    

## Bloco 4 — Empacotamento

In [ ]:
# Junta em /content/docs o que vai para o repositorio e para o relatorio.
DOCS = Path("/content/docs")

ARQUIVOS = ["weights/best.pt", "args.yaml", "results.csv", "results.png","confusion_matrix.png",
            "confusion_matrix_normalized.png", "BoxPR_curve.png",
            "BoxF1_curve.png", "BoxP_curve.png", "BoxR_curve.png", "val_batch0_pred.jpg","val_batch0_labels.jpg",
            "val_batch1_pred.jpg","val_batch1_labels.jpg", "val_batch2_pred.jpg","val_batch2_labels.jpg"]

for run in ["A_proprio", "A_val_teste", "B_publico", "B_val_teste_proprio"]:
    origem_run = Path("/content/runs")/run
    if not origem_run.exists():
        print(f"run nao encontrada: {run}")
        continue
    destino = DOCS/run
    destino.mkdir(parents=True, exist_ok=True)
    for rel in ARQUIVOS:
        if (origem_run/rel).exists():
            shutil.copy(origem_run/rel, destino/Path(rel).name)

# Os data.yaml identificam a versao de cada dataset: sem eles o run nao e
# reproduzivel.
shutil.copy(RAIZ_A/"data.yaml", DOCS/"data_A_proprio.yaml")
shutil.copy(RAIZ_PUB/"data.yaml", DOCS/"data_B_publico.yaml")

for run in ["A_video", "B_video"]:
    pasta = Path("/content/runs")/run
    if pasta.exists():
        for mp4 in pasta.glob("*.avi"):
            shutil.copy(mp4, DOCS/f"{run}.avi")

!find /content/docs -type f | sort

/content/docs/A_proprio/args.yaml
/content/docs/A_proprio/best.pt
/content/docs/A_proprio/BoxF1_curve.png
/content/docs/A_proprio/BoxP_curve.png
/content/docs/A_proprio/BoxPR_curve.png
/content/docs/A_proprio/BoxR_curve.png
/content/docs/A_proprio/confusion_matrix_normalized.png
/content/docs/A_proprio/confusion_matrix.png
/content/docs/A_proprio/results.csv
/content/docs/A_proprio/results.png
/content/docs/A_proprio/val_batch0_labels.jpg
/content/docs/A_proprio/val_batch0_pred.jpg
/content/docs/A_proprio/val_batch1_labels.jpg
/content/docs/A_proprio/val_batch1_pred.jpg
/content/docs/A_proprio/val_batch2_labels.jpg
/content/docs/A_proprio/val_batch2_pred.jpg
/content/docs/A_val_teste/BoxF1_curve.png
/content/docs/A_val_teste/BoxP_curve.png
/content/docs/A_val_teste/BoxPR_curve.png
/content/docs/A_val_teste/BoxR_curve.png
/content/docs/A_val_teste/confusion_matrix_normalized.png
/content/docs/A_val_teste/confusion_matrix.png
/content/docs/A_val_teste/val_batch0_labels.jpg
/content/docs/

In [ ]:
from google.colab import files
shutil.make_archive("/content/docs", "zip", DOCS)
files.download("/content/docs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>